# Projet 4 — TechNova Partners : Analyse d'attrition
## Notebook 01 — Chargement, jointure et exploration

---
## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Style global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', None)

# Chemins vers les fichiers
DATA_DIR = Path('data')   # adaptez selon votre arborescence
SIRH_PATH   = DATA_DIR / 'extrait_sirh.csv'
EVAL_PATH   = DATA_DIR / 'extrait_eval.csv'
SONDAGE_PATH = DATA_DIR / 'extrait_sondage.csv'

---
## 1. Chargement des fichiers sources

In [ ]:
sirh    = pd.read_csv(SIRH_PATH)
eval_df = pd.read_csv(EVAL_PATH)
sondage = pd.read_csv(SONDAGE_PATH)

print(f'SIRH    : {sirh.shape}')
print(f'Eval    : {eval_df.shape}')
print(f'Sondage : {sondage.shape}')

In [ ]:
# Aperçu rapide de chaque fichier
display(sirh.head(3))
display(eval_df.head(3))
display(sondage.head(3))

In [ ]:
# Types et valeurs manquantes par fichier
for name, frame in [('SIRH', sirh), ('Eval', eval_df), ('Sondage', sondage)]:
    print(f'\n=== {name} ===')
    print(frame.dtypes)
    missing = frame.isnull().sum()
    print('Valeurs manquantes :', missing[missing > 0].to_dict() or 'aucune')

---
## 2. Jointure des 3 fichiers

**Clés de jointure :**
- `sirh.id_employee` ↔ `sondage.code_sondage` (même valeur entière)
- `sirh.id_employee` ↔ `eval_df.eval_number` (format `"E_{id}"` → normalisation nécessaire)

In [ ]:
def normaliser_cle_eval(df: pd.DataFrame) -> pd.DataFrame:
    """Extrait l'entier depuis la clé eval_number ('E_42' → 42)."""
    df = df.copy()
    df['id_employee'] = df['eval_number'].str.replace('E_', '', regex=False).astype(int)
    return df

eval_df = normaliser_cle_eval(eval_df)
print('Clé eval normalisée — sample :', eval_df[['eval_number', 'id_employee']].head(5).to_dict('records'))

In [ ]:
# Jointure inner (les 3 fichiers couvrent la même population → inner = left ici)
df = (
    sirh
    .merge(eval_df, on='id_employee', how='inner')
    .merge(sondage, left_on='id_employee', right_on='code_sondage', how='inner')
)

print(f'DataFrame central : {df.shape}')
print(f'Colonnes ({len(df.columns)}) :', df.columns.tolist())

In [ ]:
# Vérification : aucune ligne perdue
assert len(df) == len(sirh), f'Jointure incomplète : {len(df)} lignes vs {len(sirh)} attendues'
print('✓ Jointure complète — aucune ligne perdue')

---
## 3. Nettoyage initial

In [ ]:
def nettoyer_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Nettoyage du DataFrame central :
    - Suppression des colonnes redondantes ou constantes
    - Conversion de la colonne augmentation_salaire ("11 %" → 11.0)
    """
    df = df.copy()

    # Colonnes identifiants devenus redondants après jointure
    df = df.drop(columns=['eval_number', 'code_sondage'], errors='ignore')

    # Colonnes constantes détectées (aucune information)
    #   - nombre_heures_travailless : une seule valeur (80)
    #   - ayant_enfants             : une seule valeur ('Y')
    cols_constantes = [c for c in df.columns if df[c].nunique() <= 1]
    if cols_constantes:
        print(f'  Colonnes constantes supprimées : {cols_constantes}')
        df = df.drop(columns=cols_constantes)

    # Nettoyage de "augementation_salaire_precedente" : '11 %' → 11.0
    if 'augementation_salaire_precedente' in df.columns:
        df['augementation_salaire_precedente'] = (
            df['augementation_salaire_precedente']
            .str.replace('%', '', regex=False)
            .str.strip()
            .astype(float)
        )
        df = df.rename(columns={'augementation_salaire_precedente': 'pct_augmentation_salaire'})

    return df


df = nettoyer_dataframe(df)
print(f'Shape après nettoyage : {df.shape}')
display(df.head(3))

In [ ]:
# Récapitulatif final : types et valeurs manquantes
info = pd.DataFrame({
    'dtype'    : df.dtypes,
    'n_unique' : df.nunique(),
    'missing'  : df.isnull().sum(),
    'sample'   : df.iloc[0],
})
display(info)

---
## 4. Statistiques descriptives globales

In [ ]:
# Distribution de la variable cible
target_counts = df['a_quitte_l_entreprise'].value_counts()
target_pct    = df['a_quitte_l_entreprise'].value_counts(normalize=True).mul(100).round(1)

print('=== Distribution de la cible ===')
print(pd.DataFrame({'count': target_counts, '%': target_pct}))
print(f"\n→ Déséquilibre des classes : {target_pct['Oui']:.1f}% de démissionnaires")
print("→ Nécessitera class_weight='balanced' ou SMOTE en modélisation")

In [ ]:
# Stats numériques globales
display(df.describe().T.round(2))

In [ ]:
# Comparaison démissionnaires vs restants sur les variables numériques
cols_num = df.select_dtypes(include='number').columns.tolist()
cols_num = [c for c in cols_num if c != 'id_employee']

comparaison = df.groupby('a_quitte_l_entreprise')[cols_num].mean().T.round(2)
comparaison['delta_%'] = (
    (comparaison['Oui'] - comparaison['Non']) / comparaison['Non'] * 100
).round(1)
display(comparaison.sort_values('delta_%', key=abs, ascending=False))

---
## 5. Visualisations exploratoires

In [ ]:
# --- 5.1 Variables numériques : distributions par groupe ---
vars_num_viz = [
    'age', 'revenu_mensuel', 'annees_dans_l_entreprise',
    'annee_experience_totale', 'distance_domicile_travail',
    'annees_depuis_la_derniere_promotion', 'pct_augmentation_salaire'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(vars_num_viz):
    for label, grp in df.groupby('a_quitte_l_entreprise'):
        axes[i].hist(
            grp[col], bins=25, alpha=0.6,
            label=f'{label} (n={len(grp)})', density=True
        )
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=7)

axes[-1].set_visible(False)
fig.suptitle('Distributions des variables numériques — démissionnaires vs restants', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('figures/01_distributions_numeriques.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.2 Taux d'attrition par variable catégorielle ---
def plot_taux_attrition(df: pd.DataFrame, col: str, ax: plt.Axes) -> None:
    """Barplot du taux d'attrition (%) par modalité d'une variable catégorielle."""
    taux = (
        df.groupby(col)['a_quitte_l_entreprise']
        .apply(lambda s: (s == 'Oui').mean() * 100)
        .sort_values(ascending=False)
    )
    taux.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.axhline(y=(df['a_quitte_l_entreprise'] == 'Oui').mean() * 100,
               color='tomato', linestyle='--', linewidth=1, label='Moyenne globale')
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('Taux attrition (%)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    ax.legend(fontsize=7)

vars_cat_viz = [
    'departement', 'poste', 'statut_marital', 'genre',
    'heure_supplementaires', 'frequence_deplacement', 'domaine_etude'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(vars_cat_viz):
    plot_taux_attrition(df, col, axes[i])

axes[-1].set_visible(False)
fig.suptitle('Taux d\'attrition par variable catégorielle', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('figures/02_taux_attrition_cat.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.3 Variables de satisfaction (scores 1-4) ---
vars_satisfaction = [
    'satisfaction_employee_environnement',
    'satisfaction_employee_nature_travail',
    'satisfaction_employee_equipe',
    'satisfaction_employee_equilibre_pro_perso',
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col in zip(axes, vars_satisfaction):
    plot_taux_attrition(df, col, ax)
    ax.set_title(col.replace('satisfaction_employee_', ''), fontsize=8)

fig.suptitle('Taux d\'attrition par score de satisfaction (1=faible, 4=élevé)', fontsize=11)
plt.tight_layout()
plt.savefig('figures/03_satisfaction.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.4 Heatmap de corrélation (variables numériques) ---
cols_corr = [c for c in cols_num if c != 'id_employee']
corr_matrix = df[cols_corr].corr(method='pearson')

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Matrice de corrélation de Pearson — variables numériques', fontsize=11)
plt.tight_layout()
plt.savefig('figures/04_correlation_pearson.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.5 Boxplots : revenu et ancienneté par attrition ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes, ['revenu_mensuel', 'annees_dans_l_entreprise']):
    sns.boxplot(
        data=df, x='a_quitte_l_entreprise', y=col,
        palette={'Non': 'steelblue', 'Oui': 'tomato'}, ax=ax
    )
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('A quitté l\'entreprise')

fig.suptitle('Distribution revenu et ancienneté selon l\'attrition', fontsize=11)
plt.tight_layout()
plt.savefig('figures/05_boxplots_cles.png', bbox_inches='tight')
plt.show()

---
## 6. Insights clés à retenir pour la présentation

In [ ]:
print('=== INSIGHTS CLÉS ===')
print()

# Taux global
taux_global = (df['a_quitte_l_entreprise'] == 'Oui').mean() * 100
print(f'1. Taux d\'attrition global : {taux_global:.1f}%')
print(f'   → Déséquilibre des classes à corriger en modélisation')
print()

# Âge moyen
age_g = df.groupby('a_quitte_l_entreprise')['age'].mean()
print(f'2. Âge moyen : Non={age_g["Non"]:.1f} ans  |  Oui={age_g["Oui"]:.1f} ans')
print(f'   → Les démissionnaires sont en moyenne {age_g["Non"]-age_g["Oui"]:.1f} ans plus jeunes')
print()

# Revenu
rev_g = df.groupby('a_quitte_l_entreprise')['revenu_mensuel'].mean()
print(f'3. Revenu mensuel moyen : Non={rev_g["Non"]:.0f}€  |  Oui={rev_g["Oui"]:.0f}€')
print()

# Heures supplémentaires
hs_g = df.groupby('heure_supplementaires')['a_quitte_l_entreprise'].apply(
    lambda s: (s == 'Oui').mean() * 100
)
print(f'4. Attrition avec heures sup : Oui={hs_g["Oui"]:.1f}%  |  Non={hs_g["Non"]:.1f}%')
print()

# Ancienneté
anc_g = df.groupby('a_quitte_l_entreprise')['annees_dans_l_entreprise'].mean()
print(f'5. Ancienneté moyenne : Non={anc_g["Non"]:.1f} ans  |  Oui={anc_g["Oui"]:.1f} ans')

---
## 7. Export du DataFrame central pour les notebooks suivants

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)
os.makedirs('figures', exist_ok=True)

df.to_csv('data/processed/df_central.csv', index=False)
print(f'DataFrame central exporté : data/processed/df_central.csv — {df.shape}')